<a href="https://colab.research.google.com/github/DataFriend101/Machine_Learning/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**1) Random Forest feature importance**

The paper reports that Average Position (43%) and Impressions (32%) are the most important features for predicting the health score.
*Question: *How much of this feature importance is expected because Average Position and Impressions are components of the health-score label itself?
Because the target is partly constructed from these same inputs, the model may be learning the structure of the target rather than discovering independent predictors of health.

**2) Logistic regression growth prediction**

The paper reports 71% holdout accuracy for a logistic regression separating growing from declining pages.
*Question:* Does the 80/20 holdout split separate pages by client or time, or can pages from the same client appear in both the training and test sets?
Because pages from the same client may share characteristics, a random holdout could produce an optimistic estimate of generalization. A client-grouped or time-aware split would provide a stronger test of whether the model performs on genuinely unseen clients or future observations.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Week 5 already used a client-grouped split, so pages from the same client were kept out of both the training and test sets. The **Week 5 Random Forest** achieved a measured **Precision@50 of 0.680** on that single grouped test split.

For this audit, I strengthen the validation by evaluating the same model across multiple client-grouped folds. This reduces dependence on one particular train/test split and provides a more stable estimate of performance on unseen clients.

In [2]:
# Load dataset
import pandas as pd
df = pd.read_csv("content_refresh_anonymized.csv")

In [5]:
# Recreate what we did in Week 5:
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

target = "is_declining_label"

features = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update"
]

X = df[features].copy()
y = df[target].copy()

print("Features:", features)
print("Target:", target)
print("Positive rate:", y.mean())

def precision_at_k(y_true, scores, k=50):
    scores = pd.Series(scores, index=y_true.index)
    top_k = scores.sort_values(ascending=False).head(k).index
    return y_true.loc[top_k].mean()

Features: ['impressions_90d', 'ctr', 'avg_position', 'content_age_days', 'days_since_last_update']
Target: is_declining_label
Positive rate: 0.5420666666666667


In [7]:
# Make validation with 5 client-grouped folds
from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
import numpy as np

groups = df["client_id"]

group_kfold = GroupKFold(n_splits=5)

fold_scores = []

for fold, (train_idx, test_idx) in enumerate(
    group_kfold.split(X, y, groups=groups), start=1
):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    # Impute using training data only
    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    # Same Random Forest as Week 5
    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    )

    model.fit(X_train, y_train)

    scores = model.predict_proba(X_test)[:, 1]

    precision = precision_at_k(
        y_test,
        scores,
        k=50
    )

    fold_scores.append(precision)

    print(f"Fold {fold} Precision@50: {precision:.3f}")

Fold 1 Precision@50: 0.800
Fold 2 Precision@50: 0.840
Fold 3 Precision@50: 0.600
Fold 4 Precision@50: 0.820
Fold 5 Precision@50: 0.780


In [8]:
# Summarize the 5 folds
mean_precision_50 = np.mean(fold_scores)
std_precision_50 = np.std(fold_scores)

print(f"Mean Precision@50: {mean_precision_50:.3f}")
print(f"Std Precision@50: {std_precision_50:.3f}")

Mean Precision@50: 0.768
Std Precision@50: 0.086


In [9]:
# Comparison of the before and the after
before_after = pd.DataFrame({
    "Validation": [
        "Week 5: Single client-grouped split",
        "Week 6: 5-fold client-grouped validation"
    ],
    "Precision@50": [
        0.680,
        mean_precision_50
    ]
})

before_after

,Validation,Precision@50
0,Week 5: Single client-grouped split,0.680
1,Week 6: 5-fold client-grouped validation,0.768


**Interpretation**

The **Week-5 Random Forest** achieved a **Precision@50 of 0.680** on its **single client-grouped test split**. Under **5-fold client-grouped validation**, the model achieved a mean **Precision@50 of 0.768**.

The results provide directional evidence that the model can rank declining pages across unseen clients, but performance varies by client group. The 5-fold result is a more robust summary than relying on a single held-out split

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I audited the final Week-5 feature set for **three potential forms of leakage**: label-derived features, future or overlapping time windows, and decision-derived features.

The target is `is_declining_label`, defined from `trend_direction`. For each feature, I checked whether it could contain information derived from the target or information that would only become available after the prediction point.

| Feature | Leakage risk checked | Assessment |
|---|---|---|
| `impressions_90d` | Could overlap with the period used to define `trend_direction` | Requires timeline verification |
| `ctr` | Could contain performance information from the label period | Requires timeline verification |
| `avg_position` | Could contain information from the label period | Requires timeline verification |
| `content_age_days` | Could be available before prediction | No obvious leakage identified |
| `days_since_last_update` | Historical information available before prediction | No obvious leakage identified |

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

My **original claim** was:

> The Random Forest achieved a Precision@50 of 0.68, compared with 0.44 for the Week 4 Baseline. The model therefore performed better than the simpler rule on the same test set and metric.

**Safer claim:**

> On the evaluated client-grouped test split, the Random Forest showed higher measured Precision@50 (0.680) than the Week-4 baseline (0.440). The 5-fold client-grouped evaluation produced a mean Precision@50 of 0.768, providing directional evidence that the model may be useful for decision-support when prioritizing potentially declining pages. These results are measured on this dataset and should not be interpreted as evidence of causal impact or guaranteed performance on future clients.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.